In [ ]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import sys
import glob
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
sys.path.insert(0, str(Path("../src").resolve()))
plt.ioff()
from model import GeosteeringHybridModel

In [ ]:
WINDOW_SIZE = 50
BATCH_SIZE = 256
# WICHTIG: Die exakten Werte aus dataset.py für die Ent-Skalierung
TVT_MEAN = 11000.0
TVT_STD = 2000.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Inferenz läuft auf: {device}")

# ---------------------------------------------------------------------
# 2. Bestes Modell laden
# ---------------------------------------------------------------------
model_path = Path("../src/models/best_geosteering_model.pth")
if not model_path.exists():
    raise FileNotFoundError(f"Modelldatei fehlt: {model_path.resolve()}")

model = GeosteeringHybridModel(num_features=2, window_size=WINDOW_SIZE)
model.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
model.to(device)
model.eval()
print("Modell erfolgreich geladen.")

In [ ]:
# ---------------------------------------------------------------------
# 3. Test-Dateien suchen
# ---------------------------------------------------------------------
test_dir = Path("../data/test")
test_files = list(test_dir.glob("*__horizontal_well.csv"))
print(f"Gefundene Test-Bohrlöcher: {len(test_files)}\n")

submission_data = []
start_time = time.perf_counter()
# ---------------------------------------------------------------------
# 4. Inferenz- & Plot-Schleife
# ---------------------------------------------------------------------
with torch.inference_mode():
    for file_path in tqdm(test_files, desc="Verarbeite und plotte Bohrlöcher", colour="green"):
        well_id = file_path.name.split("__")[0]
        df = pd.read_csv(file_path)

        # --- Daten Vorbereitung ---
        df["GR"] = pd.to_numeric(df["GR"], errors="coerce").ffill().bfill()
        df["Z"] = pd.to_numeric(df["Z"], errors="coerce").ffill().bfill()
        df["MD"] = pd.to_numeric(df["MD"], errors="coerce")  # MD sicherstellen für Plot

        features = df[["GR", "Z"]].to_numpy(dtype=np.float32, copy=True)
        features[:, 0] /= 150.0
        features[:, 1] /= 10000.0

        windows = np.lib.stride_tricks.sliding_window_view(features, window_shape=WINDOW_SIZE, axis=0)
        windows = np.ascontiguousarray(windows, dtype=np.float32)
        number_of_windows = len(windows)

        # --- Modell Vorhersage ---
        window_preds = []
        for start_idx in range(0, number_of_windows, BATCH_SIZE):
            end_idx = min(start_idx + BATCH_SIZE, number_of_windows)
            x_batch = torch.from_numpy(windows[start_idx:end_idx]).to(device)
            preds = model(x_batch).reshape(-1).cpu().numpy()
            window_preds.append(preds)

        window_preds = np.concatenate(window_preds).astype(np.float64)

        # --- Ent-Skalierung ---
        window_preds = (window_preds * TVT_STD) + TVT_MEAN

        predictions = np.full(len(df), np.nan, dtype=np.float64)
        predictions[WINDOW_SIZE - 1:] = window_preds
        df["TVT_pred"] = predictions

        # --- Plotting für jedes Bohrloch (ganz normal mit Matplotlib) ---
        fig, ax = plt.subplots(figsize=(12, 6))

        # Grüne Linie: Bekanntes TVT
        ax.plot(df['MD'], df['TVT_input'], color='green', linewidth=2, label='Bekanntes TVT_input')
        # Lila Linie: Modellvorhersage
        ax.plot(df['MD'], df['TVT_pred'], color='purple', linestyle='--', linewidth=2, label='Modell Vorhersage (TVT)')

        # Roter Bereich für den Blindflug ermitteln
        missing_mask = df["TVT_input"].isna()
        if missing_mask.any():
            eval_start_idx = missing_mask.idxmax()
            eval_start_md = float(df.loc[eval_start_idx, 'MD'])
            max_md = float(df['MD'].max())

            ax.axvline(x=eval_start_md, color='red', linestyle='-', alpha=0.5, label='Start Blindflug (NaN)')
            ax.axvspan(eval_start_md, max_md, color='red', alpha=0.1)

        ax.set_title(f'Geosteering Vorhersage für Bohrloch {well_id}')
        ax.set_xlabel('Measured Depth (MD)')
        ax.set_ylabel('True Vertical Thickness (TVT)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        fig.tight_layout()

        # Speichern und dieses GANZ SPEZIELLE Bild sauber aus dem Speicher werfen
        fig.savefig(f'img/inference_plot_{well_id}.png', dpi=100, bbox_inches='tight')
        plt.close(fig)

        # --- Kaggle Submission Daten sammeln ---
        for idx in df[missing_mask].index:
            submission_id = f"{well_id}_{idx}"
            tvt_value = df.loc[idx, "TVT_pred"]

            if pd.isna(tvt_value):
                tvt_value = TVT_MEAN

            submission_data.append({
                "ID": submission_id,
                "TVT": tvt_value
            })

In [ ]:
# ---------------------------------------------------------------------
# 6. Submission CSV erstellen und speichern
# ---------------------------------------------------------------------
submission_df = pd.DataFrame(submission_data)

# index=False ist bei Kaggle extrem wichtig, da sonst eine Extra-Spalte generiert wird
submission_df.to_csv("submission.csv", index=False)

total_time = time.perf_counter() - start_time
print(f"\nFertig! Die Datei 'submission.csv' wurde erfolgreich in {total_time:.1f} Sekunden erstellt.")
print(f"Anzahl der fehlenden Werte, die für Kaggle vorhergesagt wurden: {len(submission_df)}")